In [1]:
import os
import numpy as np
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import joblib
import tensorflow as tf

In [2]:
# =========================
# CONFIG
# =========================
data_dir = r"D:\Basant\Graduation Project\AI\AI-online\Final Data\BALANCED_DATA"
emergency_classes = ["Siren", "Gunshot", "Cracking"]
SAMPLE_RATE = 16000

In [3]:
# =========================
# FEATURE EXTRACTION
# =========================
def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    # حماية من silence / crash
    max_val = np.max(np.abs(audio))
    if max_val == 0:
        max_val = 1e-9

    audio = audio / max_val

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc.T, axis=0)
    mfcc_std = np.std(mfcc.T, axis=0)

    zcr = np.mean(librosa.feature.zero_crossing_rate(audio))
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
    rms = np.mean(librosa.feature.rms(y=audio))

    return np.concatenate([
        mfcc_mean,
        mfcc_std,
        [zcr, spectral_centroid, rms]
    ])

In [4]:
# =========================
# LOAD DATA
# =========================
X, y = [], []

for cls in os.listdir(data_dir):
    class_path = os.path.join(data_dir, cls)

    if not os.path.isdir(class_path):
        continue

    for file in tqdm(os.listdir(class_path), desc=cls):
        path = os.path.join(class_path, file)

        try:
            features = extract_features(path)
            X.append(features)

            y.append(1 if cls in emergency_classes else 0)

        except Exception as e:
            pass

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Emergency:", sum(y), "Normal:", len(y) - sum(y))

Car:   0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\LEGEND\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Train: 100%|██████████| 1000/1000 [00:19<00:00, 50.24it/s]

Dataset shape: (10000, 29)
Emergency: 3000 Normal: 7000


In [5]:
# =========================
# SHUFFLE (IMPORTANT)
# =========================
X, y = shuffle(X, y, random_state=42)

In [6]:
# =========================
# TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
# =========================
# SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
# =========================
# MODEL (TFLite READY)
# =========================
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

c:\Users\LEGEND\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
# =========================
# TRAIN (with Early Stopping)
# =========================

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9770 - loss: 0.0663 - val_accuracy: 0.9600 - val_loss: 0.1600
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9781 - loss: 0.0608 - val_accuracy: 0.9631 - val_loss: 0.1476
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9786 - loss: 0.0625 - val_accuracy: 0.9563 - val_loss: 0.1516
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9787 - loss: 0.0572 - val_accuracy: 0.9606 - val_loss: 0.1616
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9786 - loss: 0.0596 - val_accuracy: 0.9619 - val_loss: 0.1460
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9758 - loss: 0.0641 - val_accuracy: 0.9531 - val_loss: 0.1581
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9780 - loss: 0.0601 - val_accuracy: 0.9663 - val_loss: 0.1572
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9778 - loss: 0.0588 - val_accu

In [18]:
# =========================
# EVALUATE (FULL METRICS + THRESHOLD TUNING)
# =========================

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# =========================
# Predict probabilities
# =========================
y_pred_proba = model.predict(X_test).flatten()

# =========================
# Threshold (Emergency tuning)
# =========================
threshold = 0.3
y_pred = (y_pred_proba > threshold).astype(int)

# =========================
# Accuracy
# =========================
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", acc)

# =========================
# Classification Report
# =========================
print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Emergency"]
))

# =========================
# Confusion Matrix
# =========================
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:\n", cm)

# =========================
# Threshold Experiment (IMPORTANT 🚨)
# =========================

print("\n========================")
print("THRESHOLD EXPERIMENT")
print("========================\n")

for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    y_pred_temp = (y_pred_proba > t).astype(int)
    acc_temp = accuracy_score(y_test, y_pred_temp)

    print(f"\nThreshold = {t}")
    print(f"Accuracy = {acc_temp}")
    print(classification_report(
        y_test,
        y_pred_temp,
        target_names=["Normal", "Emergency"]
    ))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Test Accuracy: 0.9475

Classification Report:

              precision    recall  f1-score   support

      Normal       0.98      0.95      0.96      1400
   Emergency       0.89      0.94      0.92       600

    accuracy                           0.95      2000
   macro avg       0.93      0.95      0.94      2000
weighted avg       0.95      0.95      0.95      2000


Confusion Matrix:
 [[1329   71]
 [  34  566]]

THRESHOLD EXPERIMENT


Threshold = 0.5
Accuracy = 0.9515
              precision    recall  f1-score   support

      Normal       0.97      0.96      0.97      1400
   Emergency       0.92      0.92      0.92       600

    accuracy                           0.95      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.95      0.95      0.95      2000


Threshold = 0.4
Accuracy = 0.953
              precision    recall  f1-score   support

      Normal       0.97      0.96      0.97      1400
   Emergency

In [17]:
# =========================
# SAVE (FOR TFLITE STEP)
# =========================
os.makedirs("artifacts", exist_ok=True)

model.save("artifacts/emergency_model.h5")
joblib.dump(scaler, "artifacts/scaler.pkl")

print("DONE ✅ READY FOR TFLITE 🚀")

DONE ✅ READY FOR TFLITE 🚀
